In [1]:
import torch

In [135]:
def linear_q_with_scale_and_zero_point(
    r_tensor, scale, zero_point, dtype=torch.int8):

  return torch.round((r_tensor / scale) + zero_point).clamp(torch.iinfo(dtype).min, torch.iinfo(dtype).max).to(dtype)


In [136]:
def get_q_scale_and_zero_point(r_tensor, dtype=torch.int8):

  scale = ((r_tensor.min() - r_tensor.max()) / (torch.iinfo(dtype).min - torch.iinfo(dtype).max)).item()
  zero_point = (torch.iinfo(dtype).min - (r_tensor.min() / scale)).item()
  if zero_point > torch.iinfo(dtype).max:
    zero_point = torch.iinfo(dtype).max.item()
  elif zero_point < torch.iinfo(dtype).min:
    zero_point = torch.iinfo(dtype).min.item()
  else :
    zero_point = int(round(zero_point))
  return scale, zero_point

In [137]:
test_tensor=torch.tensor(
    [[191.6, -13.5, 728.6],
     [92.14, 295.5,  -184],
     [0,     684.6, 245.5]]
)

In [138]:
scale, zero_point = get_q_scale_and_zero_point(test_tensor)
quantized = linear_q_with_scale_and_zero_point(test_tensor, scale, zero_point)

In [139]:

def linear_dequantization(quantized_tensor, scale, zero_point):
  return scale * (quantized_tensor.float()-zero_point)


In [140]:
dequantized = linear_dequantization(quantized, scale, zero_point)

In [141]:
def quantization_error(tensor, dequantized_tensor):
  return (tensor - dequantized_tensor).square().mean().item()

In [142]:
quantization_error(test_tensor, dequantized)

1.5729731321334839

In [143]:
def linear_quantization(r_tensor, dtype=torch.int8):
  scale, zero_point = get_q_scale_and_zero_point(r_tensor, dtype)
  return linear_q_with_scale_and_zero_point(r_tensor, scale, zero_point), scale, zero_point

In [144]:
quantized, scale, zero_point = linear_quantization(test_tensor, dtype=torch.int8)

dequantized = linear_dequantization(quantized, scale, zero_point)
quantization_error(test_tensor, dequantized)

1.5729731321334839

In [145]:
x = torch.randn((1024, 1024))
quantized, scale, zero_point = linear_quantization(x, dtype=torch.int8)

dequantized = linear_dequantization(quantized, scale, zero_point)
quantization_error(x, dequantized)

0.00012249924475327134

In [146]:
dequantized

tensor([[-1.6106, -0.6136, -1.0354,  ...,  0.8437, -0.4218,  0.4985],
        [-1.3422, -0.6136,  1.1888,  ..., -0.3068,  0.8053, -0.9971],
        [-0.7286, -0.8437, -0.9971,  ...,  0.8437, -1.4189,  0.5752],
        ...,
        [-2.7611, -0.8053,  1.1505,  ..., -0.4602,  0.5369, -2.4927],
        [ 0.8820, -0.6519, -1.3805,  ..., -1.2272,  0.3835,  0.9971],
        [ 0.6903,  0.6519, -0.1150,  ...,  0.6136,  0.9587, -0.4218]])

In [147]:
linear_quantization(test_tensor, dtype=torch.int8)

(tensor([[ -23,  -81,  127],
         [ -51,    6, -128],
         [ -77,  114,   -8]], dtype=torch.int8),
 3.5788233280181885,
 -77)

In [162]:
def get_q_scale_symmetric(tensor, dtype=torch.int8):
  return (tensor.abs().max() / torch.iinfo(dtype).max).item()

In [163]:

get_q_scale_symmetric(test_tensor, dtype=torch.int8)

5.7370076179504395

In [164]:
def linear_q_symmetric(tensor, dtype=torch.int8):
  scale = get_q_scale_symmetric(tensor, dtype)
  return linear_q_with_scale_and_zero_point(tensor, scale, 0, dtype), scale


In [165]:
quantized, scale = linear_q_symmetric(test_tensor, dtype=torch.int8)

dequantized = linear_dequantization(quantized, scale, 0)
quantization_error(test_tensor, dequantized)

2.5091912746429443

In [166]:
def linear_q_symmetric_per_channel(r_tensor, dim, dtype=torch.int8):

  scale = torch.zeros(r_tensor.shape[dim])
  for i in range(r_tensor.shape[dim]):
    scale[i] = get_q_scale_symmetric(r_tensor.select(dim, i))
  reshape_dims = [1, 1]
  reshape_dims[dim] = -1
  scale = scale.view(reshape_dims)
  quantized = linear_q_with_scale_and_zero_point(r_tensor, scale, 0, dtype = torch.int8)
  return quantized, scale


In [167]:
quantized, scale = linear_q_symmetric_per_channel(test_tensor, dim = 0, dtype=torch.int8)
dequantized = linear_dequantization(quantized, scale, 0)
quantization_error(test_tensor, dequantized)

1.8084441423416138

In [168]:
quantized, scale = linear_q_symmetric_per_channel(test_tensor, dim = 1, dtype=torch.int8) # columns wise
dequantized = linear_dequantization(quantized, scale, 0)
quantization_error(test_tensor, dequantized)

1.0781488418579102

In [169]:
def quantization_error(tensor, dequantized):
  return (tensor - dequantized).square().mean().item()

In [170]:
def linear_q_symmetric_per_group(tensor, group_size,
                                 dtype=torch.int8):
  assert tensor.shape[1] % group_size == 0
  assert len(tensor.shape) == 2
  tensor_group = tensor.view(-1, group_size)
  quantized, scale = linear_q_symmetric_per_channel(tensor_group, dim = 0, dtype=dtype)
  return quantized.view(tensor.shape), scale

In [171]:
x = torch.randn(6, 6)
quantized, scale = linear_q_symmetric_per_group(x, 3,
                                 dtype=torch.int8)

In [172]:
def linear_dequantization_per_group(quantized, scale,
                                    group_size):
  quantized_group = quantized.view(-1, group_size)
  dequantized_group = linear_dequantization(quantized_group, scale, 0)
  return dequantized_group.view(quantized.shape)

In [173]:
dequantized = linear_dequantization_per_group(quantized, scale,
                                    3)

In [174]:
x, dequantized

(tensor([[ 0.0693, -1.3801, -0.8345, -1.7295,  0.8985, -0.0995],
         [-0.8086,  1.0520,  0.1038,  1.6718,  1.1814,  0.3152],
         [ 0.7763,  1.5666, -0.1857, -0.5505,  1.0793,  0.9220],
         [ 0.7324,  0.5379,  0.0828, -1.3986,  1.1042, -0.5454],
         [-0.6465,  0.5141,  0.9337,  1.2015,  0.9163, -1.8051],
         [ 0.0894,  0.0512, -0.3047,  0.5391,  0.2678,  1.0702]]),
 tensor([[ 0.0652, -1.3801, -0.8367, -1.7295,  0.8988, -0.0953],
         [-0.8118,  1.0520,  0.1077,  1.6718,  1.1847,  0.3159],
         [ 0.7771,  1.5666, -0.1850, -0.5524,  1.0793,  0.9178],
         [ 0.7324,  0.5363,  0.0807, -1.3986,  1.1012, -0.5506],
         [-0.6470,  0.5146,  0.9337,  1.2081,  0.9096, -1.8051],
         [ 0.0888,  0.0504, -0.3047,  0.5393,  0.2697,  1.0702]]))

In [175]:
quantization_error(x, dequantized)

6.517128440464148e-06

In [ ]:
def linear_dequantization_per_group(quantized_tensor, scale,
                                    group_size):